---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: Consumo de LLMs via API</h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
  <img src="https://img.shields.io/badge/OpenAI-412991?style=for-the-badge&logo=openai&logoColor=white"/>
</div>

---

In [ ]:
# Obs: se você não estiver utilizando um ambiente virtual, instale as bibliotecas conforme se apresenta abaixo
#%pip install -q -r requirements.txt

# pip é o gerenciador de pacotes do Python. Pense nele como o instalador oficial de libs Python.
# no notebook, usar %pip ... é ideal porque instala no mesmo ambiente do kernel em uso.

# -q: quiet
# -r: requirement file, indica ao pip para instalar os pacotes listados no arquivo requirements.txt

---

<div align="center">

## <span style="color:#1E90FF;">LLMs via API: Visão Geral</span>

</div>

Nas aulas anteriores, construímos e treinamos modelos de linguagem do zero — primeiro com PyTorch, depois apenas com NumPy. Agora daremos um passo diferente: **consumir modelos já treinados e muito mais poderosos por meio de uma API**.

Grandes Modelos de Linguagem (LLMs) como o GPT-4o da OpenAI possuem bilhões de parâmetros e foram treinados em enormes volumes de dados. Treiná-los do zero exigiria infraestrutura computacional inacessível para a maioria dos projetos. A alternativa prática é consumi-los via API: enviamos uma requisição HTTP com o texto de entrada e recebemos a resposta gerada pelo modelo.

O endpoint central para geração de texto é o **Chat Completions**, que opera sobre uma lista de mensagens — permitindo conversas com múltiplos turnos, instruções de sistema e controle fino sobre o comportamento do modelo por meio de **parâmetros de geração**.

In [1]:
%pip install openai --quiet


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---

<div align="center">

## <span style="color:#1E90FF;">Setup: Configuração do Cliente</span>

</div>

In [2]:
import os
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("Cliente OpenAI configurado com sucesso.")

---

<div align="center">

## <span style="color:#1E90FF;">Estrutura das Mensagens</span>

</div>

O Chat Completions recebe uma lista de mensagens (`messages`). Cada mensagem possui dois campos obrigatórios:

| Campo | Tipo | Descrição |
|---|---|---|
| `role` | string | Quem está falando: `system`, `user` ou `assistant` |
| `content` | string | O texto da mensagem |

### Papéis (`role`)

- **`system`** — Define o comportamento geral do modelo: personalidade, restrições, formato esperado. É processado antes de tudo e orienta como o modelo responde às mensagens seguintes.
- **`user`** — Representa a entrada do usuário: perguntas, instruções, prompts.
- **`assistant`** — Representa respostas anteriores do modelo. Usado para construir conversas com múltiplos turnos (histórico).

```
[ system ]  →  define o contexto e as regras
[ user   ]  →  envia a pergunta ou tarefa
[ assistant ] →  resposta anterior (opcional, para histórico)
[ user   ]  →  próxima pergunta
    ...
```

---

<div align="center">

## <span style="color:#1E90FF;">Primeira Chamada</span>

</div>

In [4]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Você é um assistente chamado JullesBot. Sempre diga o seu nome junto da resposta."},
        {"role": "user",   "content": "O que é um Large Language Model? Responda em 3 linhas."}
    ]
)

print(response.choices[0].message.content)

Um Large Language Model (LLM) é um tipo de inteligência artificial projetado para entender e gerar linguagem humana. Ele é treinado em vastos conjuntos de dados textuais, permitindo que produza respostas contextualmente relevantes. LLMs são utilizados em diversas aplicações, como chatbots, tradução automática e assistentes virtuais. – JullesBot.


### Anatomia da resposta

O objeto `response` retornado pela API contém muito mais do que apenas o texto gerado:

In [5]:
print("Modelo usado     :", response.model)
print("Motivo de parada :", response.choices[0].finish_reason)
print("Tokens de entrada:", response.usage.prompt_tokens)
print("Tokens de saída  :", response.usage.completion_tokens)
print("Tokens totais    :", response.usage.total_tokens)

Modelo usado     : gpt-4o-mini-2024-07-18
Motivo de parada : stop
Tokens de entrada: 47
Tokens de saída  : 72
Tokens totais    : 119


In [6]:
# motivo de parada pode ser:
# - stop: o modelo parou porque atingiu um token de parada definido (ex: \n\n)
# - length: o modelo parou porque atingiu o limite máximo de tokens definido na requisição
# - content_filter: o modelo parou porque detectou conteúdo que violava as políticas de uso
response.choices[0].finish_reason

'stop'

---

<div align="center">

## <span style="color:#1E90FF;">Parâmetros de Geração</span>

</div>

Os parâmetros de geração controlam **como** o modelo produz texto. Ajustá-los corretamente faz a diferença entre uma resposta genérica e uma resposta precisa para o contexto da aplicação.

---

### `model` — Escolha do Modelo

Define qual LLM processa a requisição. A escolha envolve trade-offs entre **capacidade**, **velocidade** e **custo**.

| Modelo | Uso recomendado |
|---|---|
| `gpt-4o-mini` | Tarefas rotineiras, alto volume, baixo custo |
| `gpt-4o` | Tarefas complexas, raciocínio, análise |
| `o3-mini` | Raciocínio matemático e lógico (chain-of-thought interno) |

> **Regra prática:** comece com `gpt-4o-mini` e escale para modelos mais capazes somente se a qualidade das respostas não for suficiente.

In [7]:
prompt_modelo = "Explique o conceito de overfitting em uma frase."

for modelo in ["gpt-4o-mini", "gpt-4o"]:
    resp = client.chat.completions.create(
        model=modelo,
        messages=[{"role": "user", "content": prompt_modelo}]
    )
    print(f"[{modelo}]")
    print(resp.choices[0].message.content)
    print(f"Tokens: {resp.usage.total_tokens}")
    print()

[gpt-4o-mini]
Overfitting é um fenômeno em aprendizado de máquina onde um modelo se ajusta excessivamente aos dados de treinamento, capturando ruídos e variações aleatórias, o que prejudica sua capacidade de generalização em novos dados.
Tokens: 67

[gpt-4o]
Overfitting é quando um modelo de aprendizado de máquina se ajusta excessivamente aos dados de treinamento, capturando ruído e flutuações aleatórias em vez de padrões subjacentes, resultando em um desempenho pior em dados novos ou não vistos.
Tokens: 73



---

### `temperature` — Criatividade vs. Determinismo

Controla a **aleatoriedade** na seleção de tokens. Tecnicamente, escala os logits antes da aplicação do softmax:

$$P(\text{token}_i) = \frac{e^{\,z_i / T}}{\sum_j e^{\,z_j / T}}$$

onde $z_i$ é o logit do token $i$ e $T$ é a temperatura.

| Valor | Efeito | Cenário de uso |
|---|---|---|
| `0.0` | Determinístico — sempre o token mais provável | Classificação, extração de dados, código |
| `0.2 – 0.5` | Muito focado, pouca variação | Resumos, respostas factuais |
| `0.7 – 1.0` | Equilibrado — padrão da maioria dos usos | Chatbots, explicações |
| `1.5 – 2.0` | Alta criatividade, saídas imprevisíveis | Brainstorming, escrita criativa |

> **Atenção:** `temperature` e `top_p` não devem ser alterados simultaneamente — ajuste apenas um deles por vez.

In [8]:
prompt_temp = "Escreva um título criativo para um artigo sobre redes neurais."

for temp in [0.0, 0.7, 1.5]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_temp}],
        temperature=temp
    )
    print(f"[temperature={temp}] {resp.choices[0].message.content.strip()}")

[temperature=0.0] "Desvendando o Futuro: Como as Redes Neurais Estão Transformando o Mundo Digital"
[temperature=0.7] "Desvendando o Labirinto Digital: Como as Redes Neurais Estão Transformando o Futuro da Tecnologia"
[temperature=1.5] "Explorando o Labirinto Digital: A Revolução das Redes Neurais na Era da Informação"


**Experimento — repetindo com `temperature=0`:**

Com temperatura zero, múltiplas execuções do mesmo prompt devem retornar a mesma resposta (ou respostas muito próximas). Com temperatura alta, a variação é grande mesmo entre execuções idênticas.

In [9]:
print("=== temperature=0.0 (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_temp}],
        temperature=0.0
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()}")

print()
print("=== temperature=1.5 (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_temp}],
        temperature=1.5
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()}")

=== temperature=0.0 (3 execuções) ===
  [1] "Desvendando o Futuro: Como as Redes Neurais Estão Transformando o Mundo Digital"
  [2] "Desvendando o Cérebro Digital: A Revolução das Redes Neurais na Era da Inteligência Artificial"
  [3] "Desvendando o Futuro: Como as Redes Neurais Estão Transformando o Mundo Digital"

=== temperature=1.5 (3 execuções) ===
  [1] "Desbravando o Intangível: Como as Redes Neurais Estão Reformulando a Inteligência Artificial"
  [2] "Desvendando o Labirinto da Inteligência: A Revolução das Redes Neurais"
  [3] "Desvendando Neuralópolis: A Revolução Inteligente das Redes Neurais"


---

### `max_tokens` — Limite de Tokens na Saída

Define o número máximo de tokens que o modelo pode gerar na resposta. Não afeta a **qualidade** — apenas trunca a saída quando o limite é atingido.

> **O que é um token?** Aproximadamente ¾ de uma palavra em inglês ou um pouco menos em português. "inteligência artificial" ≈ 4–5 tokens. Uma página de texto ≈ 500–700 tokens.

| Valor | Efeito |
|---|---|
| Muito baixo (ex.: `20`) | Resposta truncada, `finish_reason = "length"` |
| Adequado | Resposta completa, `finish_reason = "stop"` |
| Muito alto | Sem impacto se o modelo termina antes — mas aumenta o custo máximo possível |

> **Nota:** para modelos de raciocínio (`o3`, `o4-mini`), o parâmetro equivalente é `max_completion_tokens`, pois ele inclui os tokens do raciocínio interno.

In [10]:
prompt_tokens = "Explique como funciona o mecanismo de atenção em transformers."

for limite in [20, 80, 300]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_tokens}],
        max_tokens=limite
    )
    texto = resp.choices[0].message.content.strip()
    motivo = resp.choices[0].finish_reason
    print(f"[max_tokens={limite} | finish_reason={motivo}]")
    print(texto)
    print()

[max_tokens=20 | finish_reason=length]
O mecanismo de atenção é um dos componentes fundamentais da arquitetura de Transformers, que revolucionou o campo de

[max_tokens=80 | finish_reason=length]
O mecanismo de atenção é um dos componentes centrais dos modelos de Transformer, que revolucionaram o campo do processamento de linguagem natural (NLP) e outras áreas, como visão computacional. O mecanismo de atenção permite que o modelo foque em diferentes partes de uma sequência de entrada ao gerar uma saída, melhorando sua capacidade de entender contextos complexos.

### Estrutura do Mecanismo de

[max_tokens=300 | finish_reason=length]
O mecanismo de atenção, especificamente o que é utilizado em modelos de Transformers, é uma técnica fundamental que permite que o modelo processe informações de maneira mais eficiente e eficaz, considerando a relação entre diferentes partes de uma sequência de entrada. Abaixo está uma explicação detalhada de como o mecanismo de atenção funciona:

### 1. **Co

---

### `top_p` — Nucleus Sampling

Define o tamanho do conjunto de tokens candidatos à seleção. O modelo considera apenas os tokens que juntos somam probabilidade acumulada de `top_p`.

| Valor | Efeito |
|---|---|
| `1.0` (padrão) | Considera todos os tokens do vocabulário |
| `0.9` | Considera apenas os tokens que juntos representam 90% da probabilidade |
| `0.1` | Considera apenas os tokens mais prováveis (resposta muito focada) |

A diferença prática para `temperature` é sutil, mas importante:
- `temperature` escala **todos os logits** antes do softmax — muda a forma da distribuição inteira.
- `top_p` **filtra** os tokens de menor probabilidade após o softmax — mantém a distribuição original, mas só amostra do núcleo.

> **Boa prática:** ajuste `temperature` **ou** `top_p`, nunca os dois ao mesmo tempo.

In [11]:
prompt_topp = "Descreva a sensação de aprender algo novo."

# temperature=1.0 é o valor PADRÃO da API — mantemos fixo para isolar o efeito do top_p.
# "Ajustar os dois simultaneamente" significa alterar AMBOS para valores não-padrão.
# Usar temperature=1.0 (padrão) + variar top_p NÃO viola essa regra.
for tp in [0.1, 0.5, 1.0]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_topp}],
        top_p=tp,
        temperature=1.0  # padrão — mantido fixo intencionalmente para isolar top_p
    )
    print(f"[top_p={tp}]")
    print(resp.choices[0].message.content.strip())
    print()

[top_p=0.1]
Aprender algo novo é uma experiência rica e multifacetada, que pode evocar uma variedade de sensações e emoções. No início, pode haver uma mistura de curiosidade e ansiedade, especialmente se o assunto for desafiador. À medida que você se aprofunda, a sensação de descoberta se intensifica; cada nova informação ou habilidade adquirida traz uma onda de satisfação e empolgação.

O cérebro, ao assimilar novos conhecimentos, libera neurotransmissores que promovem uma sensação de prazer e recompensa. Isso pode resultar em uma espécie de "alta" mental, onde a motivação e a energia aumentam. A superação de dificuldades e a resolução de problemas também geram um sentimento de realização e autoconfiança.

Além disso, aprender algo novo pode ser um processo social, envolvendo trocas e interações com outras pessoas, o que pode criar um senso de comunidade e pertencimento. A sensação de estar em um caminho de crescimento pessoal e intelectual é profundamente gratificante, e muitas vezes

---

### `frequency_penalty` e `presence_penalty` — Controle de Repetição

Ambos os parâmetros desincentivam o modelo a repetir tokens, mas de formas distintas. Matematicamente, eles ajustam os logits antes da amostragem:

$$z_i^{\prime} = z_i - \alpha \cdot c_i - \beta \cdot \mathbb{1}[c_i > 0]$$

onde $c_i$ é o número de vezes que o token $i$ já apareceu, $\alpha$ é o `frequency_penalty` e $\beta$ é o `presence_penalty`.

| Parâmetro | Intervalo | Efeito |
|---|---|---|
| `frequency_penalty` | `-2.0` a `2.0` | Penaliza proporcionalmente à **frequência** — quanto mais o token apareceu, mais penalizado |
| `presence_penalty` | `-2.0` a `2.0` | Penaliza pela **presença** — qualquer token que apareceu uma vez já recebe a penalidade máxima |

- **`frequency_penalty` alto:** evita que o modelo repita as mesmas palavras várias vezes no mesmo texto.
- **`presence_penalty` alto:** encoraja o modelo a mudar de assunto e introduzir novos temas.
- **Valores negativos:** efeito contrário — incentivam repetição (útil em raros casos de geração estruturada).

In [ ]:
prompt_penalty = "Fale sobre inteligência artificial."

configs = [
    {"label": "Sem penalidades (padrão)",          "frequency_penalty": 0.0, "presence_penalty": 0.0},
    {"label": "Alta frequency_penalty",             "frequency_penalty": 2.0, "presence_penalty": 0.0},
    {"label": "Alta presence_penalty",              "frequency_penalty": 0.0, "presence_penalty": 2.0},
]

for cfg in configs:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_penalty}],
        max_tokens=120,
        frequency_penalty=cfg["frequency_penalty"],
        presence_penalty=cfg["presence_penalty"]
    )
    print(f"[{cfg['label']}]")
    print(resp.choices[0].message.content.strip())
    print()

---

### `stop` — Sequências de Parada

Define até 4 strings que, quando geradas, fazem o modelo parar imediatamente. Útil para formatos estruturados onde se sabe exatamente onde a saída deve terminar.

A sequência de parada **não é incluída** na resposta.

In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Responda somente com a lista solicitada, sem introdução."},
        {"role": "user",   "content": "Liste 5 aplicações de machine learning, uma por linha, numeradas."}
    ],
    stop=["3."]  # para na 3ª linha — retorna apenas os 2 primeiros itens
)

print(resp.choices[0].message.content)
print(f"\nfinish_reason: {resp.choices[0].finish_reason}")

---

### `n` — Múltiplas Respostas

Gera `n` respostas independentes para o mesmo prompt. Cada resposta fica em `response.choices[i]`.

> **Custo:** o custo é proporcional a `n` — gerar 3 respostas consome aproximadamente 3x os tokens de saída. Use com moderação.

In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Sugira um nome para uma startup de IA educacional."}],
    n=3,
    temperature=1.2
)

for i, choice in enumerate(resp.choices):
    print(f"[Opção {i+1}] {choice.message.content.strip()}")

print(f"\nTotal de tokens consumidos: {resp.usage.total_tokens}")

---

### `seed` — Reprodutibilidade

Quando fornecido, o modelo tenta gerar a mesma resposta para o mesmo prompt com os mesmos parâmetros. Útil para **testes**, **debugging** e **experimentos comparativos**.

> **Importante:** a OpenAI não garante reprodutibilidade 100% — atualizações do modelo podem alterar as respostas mesmo com o mesmo `seed`. Verifique o campo `system_fingerprint` na resposta para identificar mudanças na infraestrutura.

In [ ]:
prompt_seed = "Gere um número aleatório entre 1 e 100 e justifique sua escolha."

print("=== Com seed=42 (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_seed}],
        temperature=1.0,
        seed=42
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()[:80]}...")
    print(f"       system_fingerprint: {resp.system_fingerprint}")

print()
print("=== Sem seed (3 execuções) ===")
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_seed}],
        temperature=1.0
    )
    print(f"  [{i+1}] {resp.choices[0].message.content.strip()[:80]}...")

---

### `logprobs` — Probabilidades dos Tokens

Quando `logprobs=True`, a API retorna o **log-probabilidade** de cada token gerado. Com `top_logprobs=N` (até 20), também retorna os N tokens alternativos mais prováveis em cada posição.

> **Log-probabilidade:** `logprob = log(probabilidade)`. Um valor próximo de `0` significa alta probabilidade (quase certeza); valores muito negativos indicam baixa probabilidade. Para converter: `prob = exp(logprob)`.

**Útil para:**
- Entender a "confiança" do modelo em cada palavra escolhida
- Ver quais alternativas o modelo considerou antes de escolher
- Detectar pontos de ambiguidade na geração
- Implementar métricas de calibração e avaliação

| `logprob` | `prob` aproximada | Interpretação |
|---|---|---|
| `0.0` | 100% | Certeza absoluta |
| `-0.1` | ~90% | Alta confiança |
| `-0.7` | ~50% | Incerteza significativa |
| `-2.3` | ~10% | Token improvável |
| `< -4.6` | < 1% | Token muito improvável |

In [ ]:
import math

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "A capital do Brasil é"}],
    max_tokens=5,
    logprobs=True,
    top_logprobs=3,
    temperature=0.0
)

print("Tokens gerados e suas probabilidades:\n")
for token_info in resp.choices[0].logprobs.content:
    prob = math.exp(token_info.logprob) * 100
    print(f"  Token: {repr(token_info.token):15s} | logprob: {token_info.logprob:7.4f} | prob: {prob:6.2f}%")
    print(f"  Alternativas consideradas:")
    for alt in token_info.top_logprobs:
        alt_prob = math.exp(alt.logprob) * 100
        print(f"    {repr(alt.token):15s} → {alt_prob:.2f}%")
    print()

---

### `stream` — Respostas em Tempo Real

Quando `stream=True`, os tokens são enviados progressivamente à medida que são gerados — em vez de aguardar a resposta completa. O cliente recebe um iterador de `chunks`, cada um contendo um fragmento do texto.

**Quando usar:**
- Interfaces conversacionais (chatbots, terminais)
- Respostas longas onde a latência percebida importa
- Monitoramento em tempo real da geração

**Quando não usar:**
- Pipelines de processamento em batch (não há ganho real)
- Quando você precisa do texto completo antes de processá-lo

In [ ]:
print("Gerando resposta em streaming:\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Você é um professor paciente e detalhista."},
        {"role": "user",   "content": "Explique o que é backpropagation para um iniciante."}
    ],
    max_tokens=200,
    stream=True,
    stream_options={"include_usage": True}  # solicita estatísticas de uso no último chunk
)

resposta_completa = ""
uso = None
for chunk in stream:
    # chunks intermediários contêm o texto; o último contém usage (choices fica vazio)
    if chunk.choices and chunk.choices[0].delta.content:
        delta = chunk.choices[0].delta.content
        print(delta, end="", flush=True)
        resposta_completa += delta
    if chunk.usage:  # presente apenas no último chunk quando stream_options inclui usage
        uso = chunk.usage

print(f"\n\nTotal de caracteres gerados: {len(resposta_completa)}")
if uso:
    print(f"Tokens de entrada: {uso.prompt_tokens} | Tokens de saída: {uso.completion_tokens} | Total: {uso.total_tokens}")

---

<div align="center">

## <span style="color:#1E90FF;">Conversas com Múltiplos Turnos</span>

</div>

A API é **stateless** — ela não guarda histórico entre chamadas. Para simular uma conversa contínua, você deve enviar todo o histórico de mensagens em cada requisição.

In [ ]:
def chat(historico, mensagem_usuario, model="gpt-4o-mini", temperature=0.7):
    historico.append({"role": "user", "content": mensagem_usuario})

    resp = client.chat.completions.create(
        model=model,
        messages=historico,
        temperature=temperature
    )

    resposta = resp.choices[0].message.content
    historico.append({"role": "assistant", "content": resposta})

    return resposta, historico


historico = [
    {"role": "system", "content": "Você é um tutor de machine learning. Responda sempre de forma breve e com exemplos simples."}
]

perguntas = [
    "O que é uma rede neural?",
    "E o que são as camadas ocultas?",
    "Qual a diferença entre overfitting e underfitting?"
]

for pergunta in perguntas:
    print(f"[Usuário] {pergunta}")
    resposta, historico = chat(historico, pergunta)
    print(f"[Assistente] {resposta.strip()}")
    print(f"             (histórico: {len(historico)} mensagens)")
    print()

---

<div align="center">

## <span style="color:#1E90FF;">Function Calling (Tool Use)</span>

</div>

O **Function Calling** permite que o modelo solicite a execução de funções definidas pelo desenvolvedor quando precisa de informações externas ou ações no mundo real.

O fluxo ocorre em duas chamadas à API:

```
[1ª chamada]  Usuário + definição das ferramentas → Modelo decide chamar função
              ← retorna finish_reason="tool_calls" com nome e argumentos

[Execução]    Desenvolvedor executa a função localmente com os argumentos recebidos

[2ª chamada]  Resultado da função → Modelo gera resposta final para o usuário
```

**Casos de uso:**
- Consultar APIs externas (clima, preços, dados em tempo real)
- Executar buscas em bases de dados ou sistemas internos
- Garantir saídas com schema rígido (alternativa robusta ao `json_object`)
- Construir agentes que interagem com o ambiente

**Valores de `tool_choice`:**

| Valor | Comportamento |
|---|---|
| `"auto"` (padrão) | Modelo decide se e quando chamar uma ferramenta |
| `"none"` | Nunca chama ferramentas — responde diretamente |
| `"required"` | Sempre chama uma ferramenta (não responde diretamente) |
| `{"type": "function", "function": {"name": "..."}}` | Força uma ferramenta específica |

In [12]:
import json

# 1. Definição das ferramentas disponíveis para o modelo
tools = [
    {
        "type": "function",
        "function": {
            "name": "buscar_info_modelo",
            "description": "Retorna informações sobre um modelo de ML: arquitetura, ano e área de aplicação.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nome_modelo": {
                        "type": "string",
                        "description": "Nome do modelo (ex: 'BERT', 'ResNet', 'GPT-2')"
                    }
                },
                "required": ["nome_modelo"]
            }
        }
    }
]

# 2. Base de dados local (simulando uma API externa)
base_modelos = {
    "bert":   {"arquitetura": "Transformer (encoder)", "ano": 2018, "area": "NLP"},
    "resnet": {"arquitetura": "CNN residual",           "ano": 2015, "area": "Visão Computacional"},
    "gpt2":   {"arquitetura": "Transformer (decoder)",  "ano": 2019, "area": "Geração de texto"},
}

def buscar_info_modelo(nome_modelo: str) -> dict:
    """Função local que seria chamada após o modelo solicitar."""
    chave = nome_modelo.lower().replace("-", "").replace(" ", "")
    return base_modelos.get(chave, {"erro": f"Modelo '{nome_modelo}' não encontrado."})


# 3. Primeira chamada — modelo decide se usa a ferramenta
mensagens = [{"role": "user", "content": "Me dê detalhes técnicos sobre o modelo gpt2."}]

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=mensagens,
    tools=tools,
    tool_choice="auto"  # modelo decide quando e se usa a ferramenta
)

choice = resp.choices[0]
print(f"finish_reason: {choice.finish_reason}")

if choice.finish_reason == "tool_calls":
    tool_call = choice.message.tool_calls[0]
    nome_funcao = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"Modelo solicitou: {nome_funcao}({args})")

    # 4. Desenvolvedor executa a função localmente
    resultado = buscar_info_modelo(**args)
    print(f"Resultado da função: {resultado}")

    # 5. Envia o resultado de volta ao modelo como tool message
    mensagens.append(choice.message)           # mensagem do assistente com tool_call
    mensagens.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(resultado, ensure_ascii=False)
    })

    # 6. Segunda chamada — modelo gera resposta final com os dados recebidos
    resp_final = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=mensagens,
        tools=tools
    )
    print(f"\nResposta final:\n{resp_final.choices[0].message.content}")

finish_reason: tool_calls
Modelo solicitou: buscar_info_modelo({'nome_modelo': 'GPT-2'})
Resultado da função: {'arquitetura': 'Transformer (decoder)', 'ano': 2019, 'area': 'Geração de texto'}

Resposta final:
O modelo GPT-2 possui as seguintes características técnicas:

- **Arquitetura:** Transformer (decoder)
- **Ano de Lançamento:** 2019
- **Área de Aplicação:** Geração de texto

Se precisar de mais informações ou detalhes específicos, fique à vontade para perguntar!


### Múltiplas Ferramentas e Chamadas Paralelas

Em um único response, o modelo pode solicitar **várias funções ao mesmo tempo** (`parallel_tool_calls`). Isso é mais eficiente do que chamadas sequenciais — o desenvolvedor executa todas as funções em paralelo e retorna os resultados juntos.

O padrão de loop para lidar com múltiplas ferramentas:

```
1ª chamada → finish_reason = "tool_calls"
              choice.message.tool_calls = [tc1, tc2, tc3, ...]
                                           ↓
              para cada tc: executa função → adiciona tool message
                                           ↓
2ª chamada → modelo compõe resposta final com todos os resultados
```

> **Despachador (`dispatcher`):** um dicionário `{nome_função: função}` é o padrão mais comum para rotear as chamadas dinamicamente, sem precisar de `if/elif` por função.

In [13]:
import json

# Múltiplas ferramentas + chamadas paralelas
# O modelo pode solicitar VÁRIAS funções ao mesmo tempo em um único response

tools_multiplas = [
    {
        "type": "function",
        "function": {
            "name": "obter_temperatura",
            "description": "Retorna a temperatura atual de uma cidade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cidade": {"type": "string", "description": "Nome da cidade"}
                },
                "required": ["cidade"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "obter_populacao",
            "description": "Retorna a população estimada de uma cidade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cidade": {"type": "string", "description": "Nome da cidade"}
                },
                "required": ["cidade"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "converter_moeda",
            "description": "Converte um valor de uma moeda para outra.",
            "parameters": {
                "type": "object",
                "properties": {
                    "valor":        {"type": "number", "description": "Valor a converter"},
                    "de":           {"type": "string", "description": "Moeda de origem (ex: BRL)"},
                    "para":         {"type": "string", "description": "Moeda de destino (ex: USD)"}
                },
                "required": ["valor", "de", "para"]
            }
        }
    }
]

# Implementações locais (simulando dados reais)
def obter_temperatura(cidade: str) -> dict:
    dados = {"São Paulo": 22, "Rio de Janeiro": 28, "Curitiba": 15}
    temp = dados.get(cidade, 20)
    return {"cidade": cidade, "temperatura_celsius": temp}

def obter_populacao(cidade: str) -> dict:
    dados = {"São Paulo": 12_325_000, "Rio de Janeiro": 6_748_000, "Curitiba": 1_948_000}
    pop = dados.get(cidade, 0)
    return {"cidade": cidade, "populacao": pop}

def converter_moeda(valor: float, de: str, para: str) -> dict:
    taxas = {("BRL", "USD"): 0.18, ("USD", "BRL"): 5.55, ("BRL", "EUR"): 0.17}
    taxa = taxas.get((de, para), 1.0)
    return {"valor_original": valor, "de": de, "para": para, "valor_convertido": round(valor * taxa, 2)}

despachador = {
    "obter_temperatura": obter_temperatura,
    "obter_populacao":   obter_populacao,
    "converter_moeda":   converter_moeda,
}

# Pergunta que demanda múltiplas ferramentas em paralelo
mensagens = [{
    "role": "user",
    "content": "Qual a temperatura e a população de São Paulo? E quanto é R$ 500 em dólares?"
}]

# 1ª chamada — modelo identifica e solicita todas as funções necessárias de uma vez
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=mensagens,
    tools=tools_multiplas,
    tool_choice="auto"
)

choice = resp.choices[0]
print(f"finish_reason: {choice.finish_reason}")
print(f"Número de tool_calls solicitadas: {len(choice.message.tool_calls)}\n")

# 2. Executa todas as funções solicitadas e coleta resultados
mensagens.append(choice.message)  # adiciona a mensagem do assistente com os tool_calls

for tc in choice.message.tool_calls:
    nome   = tc.function.name
    args   = json.loads(tc.function.arguments)
    print(f"  → Executando: {nome}({args})")
    resultado = despachador[nome](**args)

    mensagens.append({
        "role":         "tool",
        "tool_call_id": tc.id,
        "content":      json.dumps(resultado, ensure_ascii=False)
    })

# 3. Segunda chamada — modelo compõe resposta final com todos os resultados
resp_final = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=mensagens,
    tools=tools_multiplas
)
print(f"\nResposta final:\n{resp_final.choices[0].message.content}")

finish_reason: tool_calls
Número de tool_calls solicitadas: 3

  → Executando: obter_temperatura({'cidade': 'São Paulo'})
  → Executando: obter_populacao({'cidade': 'São Paulo'})
  → Executando: converter_moeda({'valor': 500, 'de': 'BRL', 'para': 'USD'})

Resposta final:
A temperatura atual em São Paulo é de 22°C. A população estimada da cidade é de aproximadamente 12.325.000 habitantes. 

Quanto à conversão de R$ 500, isso equivale a cerca de $90.00 dólares.


---

<div align="center">

## <span style="color:#1E90FF;">Boas Práticas</span>

</div>

### 1. System Prompt: invista na instrução de sistema

O `system` é o principal alavancador de qualidade. Um system prompt bem escrito reduz alucinações, enforça formato e elimina a necessidade de ajustes finos nos demais parâmetros.

**Elementos de um bom system prompt:**
- **Persona:** quem o modelo deve ser
- **Escopo:** o que pode e o que não pode responder
- **Formato:** como estruturar a resposta (markdown, JSON, listas...)
- **Tom:** formal, coloquial, técnico, educacional
- **Restrições:** idioma, tamanho, referências proibidas

In [ ]:
system_fraco = "Você é um assistente."

system_forte = """
Você é um assistente especializado em machine learning para estudantes de graduação.

Regras:
- Responda sempre em português brasileiro.
- Use linguagem acessível, sem jargões desnecessários.
- Quando relevante, inclua um exemplo prático curto.
- Limite suas respostas a no máximo 150 palavras.
- Se a pergunta não for sobre machine learning ou IA, diga educadamente que não pode ajudar com isso.
""".strip()

pergunta = "O que é regularização L2?"

for label, sysprompt in [("System fraco", system_fraco), ("System forte", system_forte)]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": sysprompt},
            {"role": "user",   "content": pergunta}
        ],
        temperature=0.5
    )
    print(f"[{label}]")
    print(resp.choices[0].message.content.strip())
    print()

### 2. Few-Shot Prompting: exemplos no histórico de mensagens

Incluir exemplos de entrada/saída diretamente no histórico (`user` + `assistant`) é uma das técnicas mais eficazes para guiar o comportamento do modelo — especialmente quando a instrução textual sozinha não é suficiente.

```
[ system    ]  instrução geral
[ user      ]  exemplo de entrada 1      ← few-shot
[ assistant ]  exemplo de saída 1        ← few-shot
[ user      ]  exemplo de entrada 2      ← few-shot
[ assistant ]  exemplo de saída 2        ← few-shot
[ user      ]  pergunta real do usuário
```

**Quando usar few-shot:**
- Formatos de saída rígidos que são difíceis de descrever textualmente
- Tom ou estilo muito específico
- Classificações com categorias que precisam de calibração
- Tarefas onde "mostrar" é mais claro que "descrever"

> **Custo:** cada exemplo few-shot consome tokens de entrada. Avalie o trade-off entre qualidade e custo — geralmente 2–5 exemplos já são suficientes.

In [ ]:
# Few-shot: exemplos de entrada/saída inseridos no histórico antes da pergunta real
mensagens_few_shot = [
    {"role": "system",    "content": "Você é um classificador de sentimento. Responda apenas: Positivo, Negativo ou Neutro."},
    # --- exemplos (few-shot) ---
    {"role": "user",      "content": "Adorei o produto, chegou rápido e funcionou perfeitamente!"},
    {"role": "assistant", "content": "Positivo"},
    {"role": "user",      "content": "O atendimento foi péssimo, fiquei esperando horas."},
    {"role": "assistant", "content": "Negativo"},
    {"role": "user",      "content": "O pacote chegou no prazo previsto."},
    {"role": "assistant", "content": "Neutro"},
    # --- pergunta real ---
    {"role": "user",      "content": "O modelo respondeu algumas perguntas, mas errou nas mais difíceis."}
]

resp_few = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=mensagens_few_shot,
    temperature=0.0
)
print(f"Few-shot  → {resp_few.choices[0].message.content.strip()}")

# Comparação com zero-shot no mesmo prompt
prompt_teste = "O modelo respondeu algumas perguntas, mas errou nas mais difíceis."
resp_zero = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Você é um classificador de sentimento. Responda apenas: Positivo, Negativo ou Neutro."},
        {"role": "user",   "content": prompt_teste}
    ],
    temperature=0.0
)
print(f"Zero-shot → {resp_zero.choices[0].message.content.strip()}")
print()

# Segundo exemplo: few-shot para formato customizado de output
print("--- Few-shot para formato estruturado ---")
msgs_formato = [
    {"role": "system",    "content": "Você explica conceitos de ML em formato fixo."},
    {"role": "user",      "content": "Explique: Gradient Descent"},
    {"role": "assistant", "content": "CONCEITO: Gradient Descent\nANALOGIA: Descer uma montanha no escuro seguindo a inclinação do terreno sob seus pés.\nNÍVEL: Intermediário"},
    {"role": "user",      "content": "Explique: Dropout"},
    {"role": "assistant", "content": "CONCEITO: Dropout\nANALOGIA: Treinar para uma prova estudando com diferentes combinações de anotações, nunca todas juntas.\nNÍVEL: Intermediário"},
    # pergunta real
    {"role": "user",      "content": "Explique: Batch Normalization"},
]
resp_fmt = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=msgs_formato,
    temperature=0.0
)
print(resp_fmt.choices[0].message.content.strip())

### 2. Controle de Custos: monitore o uso de tokens

In [ ]:
# Preços do gpt-4o-mini (referência: março/2026 — verifique https://openai.com/pricing para valores atualizados)
PRECO_INPUT_POR_MILHAO  = 0.150  # USD por 1M tokens de entrada
PRECO_OUTPUT_POR_MILHAO = 0.600  # USD por 1M tokens de saída

def estimar_custo(usage):
    custo_input  = (usage.prompt_tokens     / 1_000_000) * PRECO_INPUT_POR_MILHAO
    custo_output = (usage.completion_tokens / 1_000_000) * PRECO_OUTPUT_POR_MILHAO
    return custo_input + custo_output

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Você é um assistente conciso."},
        {"role": "user",   "content": "Quais são as principais diferenças entre CNN e RNN?"}
    ],
    max_tokens=200
)

uso = resp.usage
custo = estimar_custo(uso)

print(f"Tokens de entrada : {uso.prompt_tokens}")
print(f"Tokens de saída   : {uso.completion_tokens}")
print(f"Total de tokens   : {uso.total_tokens}")
print(f"Custo estimado    : USD ${custo:.6f}")
print()
print(resp.choices[0].message.content.strip())

### 3. Tratamento de Erros

A API pode retornar erros por diversas razões: limite de taxa (`RateLimitError`), chave inválida (`AuthenticationError`), conteúdo bloqueado (`BadRequestError`), indisponibilidade (`APIStatusError`). Trate-os explicitamente.

In [ ]:
from openai import RateLimitError, AuthenticationError, BadRequestError, APIStatusError

def chamar_api_com_retry(messages, model="gpt-4o-mini", max_tentativas=3, **kwargs):
    for tentativa in range(1, max_tentativas + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                **kwargs
            )
            return resp.choices[0].message.content

        except RateLimitError:
            espera = 2 ** tentativa
            print(f"  [Tentativa {tentativa}] Rate limit atingido. Aguardando {espera}s...")
            time.sleep(espera)

        except AuthenticationError:
            print("  Erro de autenticação: verifique a OPENAI_API_KEY no arquivo .env")
            return None

        except BadRequestError as e:
            print(f"  Requisição inválida: {e}")
            return None

        except APIStatusError as e:
            print(f"  Erro na API (status {e.status_code}): {e.message}")
            return None

    print("  Número máximo de tentativas atingido.")
    return None


resultado = chamar_api_com_retry(
    messages=[{"role": "user", "content": "Defina aprendizado por reforço em uma frase."}],
    temperature=0.3
)
print(resultado)

### 4. Saídas Estruturadas com JSON

Quando o modelo precisa retornar dados estruturados (para integração com outros sistemas), use `response_format` com `json_object` e instrua o modelo no system prompt.

In [ ]:
import json

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "Você extrai informações de textos e retorna sempre um JSON válido "
                "com os campos: conceito (string), area (string), dificuldade (string: basico|intermediario|avancado)."
            )
        },
        {
            "role": "user",
            "content": "Gradient descent é um algoritmo de otimização usado para minimizar funções de perda em modelos de machine learning."
        }
    ],
    response_format={"type": "json_object"},
    temperature=0.0
)

dados = json.loads(resp.choices[0].message.content)
print(json.dumps(dados, ensure_ascii=False, indent=2))

---

<div align="center">

## <span style="color:#1E90FF;">Resumo dos Parâmetros</span>

</div>

| Parâmetro | Padrão | Intervalo | Quando ajustar |
|---|---|---|---|
| `model` | — | modelos disponíveis | Sempre — escolha conforme custo e complexidade |
| `temperature` | `1.0` | `0.0` – `2.0` | Tarefas criativas (↑) ou determinísticas (↓) |
| `max_tokens` | sem limite | `1` – limite do modelo | Controlar custo máximo e evitar respostas longas |
| `top_p` | `1.0` | `0.0` – `1.0` | Alternativa ao `temperature` para focar nas palavras mais prováveis |
| `frequency_penalty` | `0.0` | `-2.0` – `2.0` | Evitar repetição de palavras no texto gerado |
| `presence_penalty` | `0.0` | `-2.0` – `2.0` | Encorajar diversidade de tópicos |
| `stop` | `null` | até 4 strings | Formatos estruturados com terminador conhecido |
| `n` | `1` | `≥ 1` | Gerar múltiplas alternativas para avaliação |
| `seed` | `null` | inteiro | Testes comparativos e reprodutibilidade |
| `logprobs` | `false` | `true/false` | Inspecionar confiança do modelo por token |
| `top_logprobs` | `null` | `1` – `20` | Ver tokens alternativos considerados em cada posição |
| `stream` | `false` | `true/false` | Interfaces em tempo real |
| `stream_options` | `null` | `{"include_usage": true}` | Capturar usage stats em modo streaming |
| `response_format` | `null` | `{"type": "json_object"}` | Forçar saída JSON válida |
| `tools` | `null` | lista de funções | Habilitar function calling |
| `tool_choice` | `"auto"` | `"auto"`, `"none"`, `"required"` | Controlar quando o modelo usa ferramentas |

---

### Guia rápido por tipo de aplicação

| Aplicação | Parâmetros recomendados |
|---|---|
| Chatbot educacional | `temperature=0.7`, system prompt detalhado |
| Extração de dados / classificação | `temperature=0.0`, `response_format=json_object` |
| Geração de código | `temperature=0.2`, `frequency_penalty=0.1` |
| Brainstorming / escrita criativa | `temperature=1.2–1.5`, `presence_penalty=0.5` |
| Resumo de documentos | `temperature=0.3`, `max_tokens` calibrado |
| Interface conversacional com streaming | `stream=True`, `stream_options={"include_usage": True}` |
| Testes e experimentos A/B | `seed` fixo, `n≥2` |
| Agentes e automações | `tools`, `tool_choice="auto"` |
| Análise de confiança do modelo | `logprobs=True`, `top_logprobs=5` |
| Few-shot para formato customizado | exemplos no histórico + `temperature=0.0` |